# PCMCI for Finance: From Causal Graph to Tradable Signal

This notebook is the **applied companion** to [`PCMCI_Intro.ipynb`](./PCMCI_Intro.ipynb).
That notebook explains the algorithm in general. This one answers the questions a
quant analyst actually has:

1. **What do the variables, lags, and edges *mean* when the data is a market?**
2. **How do I prepare price data so PCMCI gives trustworthy answers?**
3. **What signal does the [`Proof-Of-Concept.py`](../Proof-Of-Concept.py) actually compute, and why is it built that way?**
4. **How do I turn that signal into positions — and what will break in real data?**

We re-use the exact synthetic generator from the POC so every claim here is runnable.

---
## Table of Contents
1. [Why causality (not correlation) for markets](#why)
2. [Mapping markets onto a time-series DAG](#mapping)
3. [Preparing financial data for PCMCI](#prep)
4. [Running PCMCI on the panel](#run)
5. [From parents to the causal-momentum signal](#signal)
6. [Turning the signal into positions](#use)
7. [Finance-specific pitfalls](#pitfalls)
8. [Where this plugs into `quantt-pipeline`](#pipeline)

---
## 1. Why causality, not correlation <a name="why"></a>

A correlation matrix of returns is the core of portfolio theory — but it
is **symmetric and contemporaneous**. It tells you AAPL and MSFT move together; it
cannot tell you whether a move in oil *leads* airlines, or whether the 10Y yield
*drives* bank stocks or merely co-moves with them.

For signal generation we want the **directional, lagged** question:

> Does a move in variable $i$ at day $t-\tau$ give us information about variable $j$
> at day $t$ that we did **not** already have from $j$'s own history and from the
> other things driving $j$?

That "did not already have" clause is the whole game. Two reasons a naive lead-lag
correlation lies to you in markets:

- **Autocorrelation / momentum.** Returns have their own short-horizon persistence.
  A lagged correlation between oil$(t-2)$ and an airline$(t)$ can be entirely an
  artifact of each series correlating with its own past.
- **Common drivers.** "Risk-on" days push *everything* up together. Stock A at
  $t-1$ looks like it "predicts" stock B at $t$, when really a shared macro factor
  moved both. This is the indirect-path problem ($A \leftarrow M \rightarrow B$).

PCMCI's **MCI test** conditions away both of these (the source's own parents and the
target's parents), so a surviving edge is a candidate for genuine, *incremental*
predictive structure — exactly what an alpha signal needs.

---
## 2. Mapping markets onto a time-series DAG <a name="mapping"></a>

PCMCI_Intro carries over so give it a read, here's a brief dictionary to help explain the later parts 


| Abstract PCMCI object | What it is in our equity problem |
|---|---|
| Variable $X^j$ | A **return series**: one stock, an ETF, or a macro factor (examples: oil, 10Y yield, USD, VIX). |
| Time index $t$ | A **trading day**. |
| Lag $\tau$ | **Trading days of delay**. $\tau=2$ = "two sessions later". |
| Edge $X^i_{t-\tau}\to X^j_t$ | "$i$'s move $\tau$ days ago carries incremental info about $j$ today." |
| Parents $\hat{\mathcal P}(X^j)$ | The small set of lagged series that survive pruning as drivers of $j$. |
| `val_matrix[i,j,τ]` | **MCI partial correlation** — signed strength of the edge after conditioning. |

Two modelling choices matter for finance:

- **Self-edges are expected and mostly uninteresting.** A stock's own lag surviving
  is just autocorrelation/momentum. We keep these out of the *cross-asset* signal
  (see §5) but they are not bugs — the POC even counts them as a sanity check.
- **$\tau_{max}$ is a domain prior.** At daily frequency, alpha from cross-asset
  lead-lag lives in the 1–10 day range; beyond ~20 days you mostly add multiple-
  testing burden. The POC uses $\tau_{max}=5$ (one trading week).

---
## 3. Preparing financial data for PCMCI <a name="prep"></a>

PCMCI's assumptions (weak stationarity, causal sufficiency, a correctly specified CI
test) are where most finance applications quietly fail. The prep steps below are not
cosmetic — each one maps to an assumption.

| Step | Why it matters | Assumption it protects |
|---|---|---|
| **Use returns, not prices** | Price levels are non-stationary (random-walk / unit root). ParCorr on levels finds spurious links everywhere. | Stationarity |
| Prefer **log returns** | Additive across time, better-behaved tails than simple returns. | Linearity of ParCorr |
| **Align the calendar** | Different tickers have different missing days (holidays, halts). Misalignment fabricates lead-lag. | Correct lag semantics |
| **Roll the window** | Markets are regime-switching; a single 10-year fit blends incompatible regimes. Fit on a trailing window (POC: 250 days ≈ 1yr). | Local stationarity |
| **Standardize per window** | Puts all series on comparable scale so the PC pruning thresholds behave. | CI-test calibration |
| **Winsorize extreme jumps** | A single 4-sigma earnings gap can dominate a 250-pt ParCorr. | Robustness |

The "roll the window" point is the big one: **you do not fit PCMCI once.** You refit
on the trailing window every rebalance, so the causal graph — and therefore the
signal — *adapts* as relationships appear and decay. That trailing-window refit is
the POC's `WINDOW = 250` and `window_data = data[-WINDOW:]`.

In [1]:
# ── Real market panel via yfinance (replaces the synthetic generator) ─────────
# We pull daily bars for a few macro factors + a basket of single names, align
# the trading calendar across all of them, and convert to LOG RETURNS so the
# panel is (approximately) stationary — exactly the §3 prep steps, applied to
# live data. The column ORDER (macro first, then stocks) defines the indices
# that every downstream cell relies on.
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
warnings.filterwarnings("ignore")

RNG_SEED = 42
WINDOW, TAU_MIN, TAU_MAX = 250, 1, 5
ALPHA, PC_ALPHA = 0.05, 0.05

# Macro factors first, then stocks. Order == column index in `data`.
MACRO_TICKERS = ["CL=F", "^TNX", "DX-Y.NYB", "^VIX"]     # oil, 10Y yield, USD, VIX
STOCK_TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "DAL", "WMT", "NVDA", "GS"]
TICKERS       = MACRO_TICKERS + STOCK_TICKERS

START, END = "2021-01-01", "2024-01-01"

# Download adjusted closes; align the calendar by dropping any day a ticker is missing.
raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)["Close"]
prices = raw[TICKERS].dropna(how="any")                 # keep our macro->stock ordering

# Log returns -> stationary. This is the entire bridge from prices to PCMCI input.
log_rets = np.log(prices).diff().dropna()

var_names = ["oil", "yield_10y", "usd", "vix"] + [t.lower() for t in STOCK_TICKERS]
data      = log_rets.values

N_MACRO   = len(MACRO_TICKERS)
N_STOCKS  = len(STOCK_TICKERS)
N_VARS    = N_MACRO + N_STOCKS
STOCK_IDX = list(range(N_MACRO, N_VARS))

# Real data has no ground truth, so the downstream recovery check becomes a no-op.
PLANTED_EDGES = []

print("Panel shape (days, series):", data.shape)
print("Date range:", prices.index.min().date(), "->", prices.index.max().date())
print("\nEach column is a daily LOG-RETURN series (macro factors, then stocks):")
print("  macro:", var_names[:N_MACRO])
print("  stock:", var_names[N_MACRO:])
print("\nWith real data there is no planted truth — PCMCI's output is the discovery,")
print("not a recovery test. We refit on the trailing window in §4.")

/Users/benjamingorenc/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Panel shape (days, series): (752, 12)
Date range: 2021-01-04 -> 2023-12-29

Each column is a daily LOG-RETURN series (macro factors, then stocks):
  macro: ['oil', 'yield_10y', 'usd', 'vix']
  stock: ['aapl', 'msft', 'jpm', 'xom', 'dal', 'wmt', 'nvda', 'gs']

With real data there is no planted truth — PCMCI's output is the discovery,
not a recovery test. We refit on the trailing window in §4.


---
## 4. Running PCMCI on the panel <a name="run"></a>

We fit on the **trailing window only** (`data[-WINDOW:]`), mirroring how you'd refit
each rebalance. Three knobs carry finance meaning:

- **`ParCorr`** — linear partial correlation. Right default for returns: fast, and
  cross-asset lead-lag is mostly linear at daily frequency. Swap to `GPDC`/`CMIknn`
  only if you have a specific nonlinear hypothesis and enough data to support it.
- **`pc_alpha` (Phase-1 pruning)** — how aggressively to drop candidate parents.
  Looser = more candidates survive to the MCI stage.
- **BH-FDR correction** — *essential* in finance. With $N^2\cdot\tau_{max}$ tests you
  will find "significant" edges by chance. False-Discovery-Rate control on the
  p-matrix is what keeps your graph from being noise. The POC applies `fdr_bh`.

In [2]:
# ── Fit PCMCI on the trailing window, then FDR-correct the p-values ──────────
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

window_data = data[-WINDOW:]                       # refit on the recent regime
dataframe = pp.DataFrame(window_data, var_names=var_names)
pcmci = PCMCI(dataframe=dataframe,
              cond_ind_test=ParCorr(significance="analytic"), verbosity=0)

results = pcmci.run_pcmci(tau_min=TAU_MIN, tau_max=TAU_MAX, pc_alpha=PC_ALPHA)

# Multiple-testing correction: with N^2 * tau_max tests this is non-negotiable.
q_matrix = pcmci.get_corrected_pvalues(
    p_matrix=results["p_matrix"], fdr_method="fdr_bh",
    tau_min=TAU_MIN, tau_max=TAU_MAX,
)
p_matrix, val_matrix = results["p_matrix"], results["val_matrix"]
print("Fitted. val_matrix indexed as [source i, target j, lag tau], shape:",
      val_matrix.shape)

Fitted. val_matrix indexed as [source i, target j, lag tau], shape: (12, 12, 6)


In [9]:
# ── Collect surviving edges (q < ALPHA) ─────────────────────────────────────
discovered = []
for i in range(N_VARS):
    for j in range(N_VARS):
        for tau in range(TAU_MIN, TAU_MAX + 1):
            if q_matrix[i, j, tau] < ALPHA:
                discovered.append((i, j, tau, val_matrix[i, j, tau], p_matrix[i, j, tau]))

print("Discovered causal edges (post BH-FDR):")
for i, j, tau, mci, p in discovered:
    kind = "self " if i == j else "CROSS"
    print(f"  [{kind}] {var_names[i]}(t-{tau}) --MCI={mci:+.3f} (p={p:.4f})--> {var_names[j]}(t)")

# Sanity check vs planted truth — the finance analogue of a unit test.
discovered_set = {(i, j, tau) for (i, j, tau, _, _) in discovered}
print("\nRecovery check:")
for src, tgt, lag, coef in PLANTED_EDGES:
    ok = (src, tgt, lag) in discovered_set
    print(f"  {'FOUND ' if ok else 'MISSED'} {var_names[src]}(t-{lag}) -> {var_names[tgt]}(t)")

Discovered causal edges (post BH-FDR):

Recovery check:


### How to *read* one of these edges as a trader

```
[CROSS] macro_1(t-2) --MCI=+0.42 (p=0.0003)--> stock_2(t)
```

- **Direction & lag:** macro_1's move *two sessions ago* carries info about stock_2 today.
- **Sign of MCI (+0.42):** the relationship is positive — macro_1 up → stock_2 up two
  days later, *after* removing stock_2's own momentum and macro_1's persistence.
- **Magnitude:** |MCI| is a conditional partial correlation, so it doubles as a
  natural **confidence weight** for the signal (stronger, cleaner edge → bigger bet).
- **`self` edges:** a stock pointing at its own lag is plain autocorrelation. Useful
  as a sanity check, but we deliberately exclude it from the *cross-asset* alpha.

---
## 5. From parents to the causal-momentum signal <a name="signal"></a>

This is the heart of it — and exactly what step 4 of the POC computes. The logic:

> If a set of **parents** causally drive stock $s$ at known lags, and those parents
> just moved, then stock $s$ is *likely to move next* in the direction implied by
> each edge's sign and strength.

For each stock $s$, let $\mathcal P(s)$ be its **cross-causal parents** (self-edges
excluded). The signal is the MCI-weighted sum of each parent's recently *realized*
return over the edge's lag window:

$$\text{score}_s \;=\; \sum_{(i,\tau)\,\in\,\mathcal P(s)} \underbrace{\text{MCI}_{i\to s,\tau}}_{\text{edge weight}}\;\cdot\;\underbrace{\sum_{k=t-\tau+1}^{t} r_{i,k}}_{\text{parent's realized move over the lag}}$$

Why each piece is built the way it is:

- **MCI as the weight** — a stronger, cleaner causal edge should contribute more.
  The sign of MCI also flips the contribution (a negative edge means a parent rally
  is *bearish* for the child).
- **Realized parent return over `lag` bars** — the edge says the effect lands `lag`
  days later, so we sum the parent's actual return over exactly that window. That
  move is **already in the past**, so the score is computable today with no peeking.
- **Self-edges excluded** — own-momentum is a *different* signal; mixing it in would
  double-count what a standalone momentum factor already captures.

In [3]:
# ── Compute the causal-momentum score per stock (POC step 4) ─────────────────
parents_by_target = {s: [] for s in STOCK_IDX}
for i, j, tau, mci, p in discovered:
    if j in parents_by_target and i != j:          # cross-causal parents only
        parents_by_target[j].append((i, tau, mci))

latest = data.shape[0] - 1
print(f"{'stock':<10} {'score':>10}   parents used")
print("-" * 50)
scores = {}
for s in STOCK_IDX:
    parents = parents_by_target[s]
    if not parents:
        print(f"{var_names[s]:<10} {'n/a':>10}   (no cross-causal parent)")
        continue
    score, parts = 0.0, []
    for src, lag, mci in parents:
        parent_ret = data[latest - lag + 1: latest + 1, src].sum()  # realized, past
        score += mci * parent_ret
        parts.append(f"{var_names[src]}(t-{lag})")
    scores[s] = score
    print(f"{var_names[s]:<10} {score:>10.4f}   ({', '.join(parts)})")

NameError: name 'discovered' is not defined

Notice that **only the stocks with a discovered parent get a score** — the others
are `n/a`. That is a feature: PCMCI gives you a *sparse* signal that fires only where
there is statistical evidence of an external driver. You are not forced to take a
view on every name.

---
## 6. Turning the signal into positions <a name="use"></a>

The raw `score` is not a position. The standard quant workflow to get from one to the
other:

1. **Cross-sectional standardize.** Each rebalance, z-score the scores across the
   universe: $z_s = (\text{score}_s - \mu)/\sigma$. This makes the signal
   dollar-neutral-friendly and comparable across days/regimes.
2. **Map to weights.** Simplest: $w_s \propto z_s$, then scale so $\sum |w_s| = 1$
   (or to a target gross exposure). Long the high-z names, short the low-z names.
3. **Rebalance on the refit cadence.** You refit PCMCI on the trailing window, so the
   natural rebalance frequency is *weekly* (the graph won't change much intraday and
   refitting daily over-trades on noise).
4. **Risk overlay.** Cap per-name weight, cap sector/factor exposure, and size by
   the inverse of recent volatility so one volatile name doesn't dominate.

A minimal sketch (pseudocode — wire real returns in via `quantt-pipeline`):

```python
import pandas as pd
s = pd.Series(scores)                       # {stock_idx: causal-momentum score}
z = (s - s.mean()) / s.std(ddof=0)          # cross-sectional standardize
w = z / z.abs().sum()                       # dollar-neutral-ish weights, gross = 1
w = w.clip(-0.10, 0.10)                     # per-name cap
# -> hold w until the next weekly refit, then recompute from a fresh graph
```

**Backtest discipline:** because the score uses only *realized past* parent returns
and an MCI fit on *past* data, it is point-in-time by construction — but you must
still lag your *execution* (trade at next open, not the close you computed on) and
charge realistic costs. Cross-asset lead-lag edges are often small and decay fast.

### How this complements the `quantt-pipeline` signals

The pipeline's `features.py` already ships **momentum (`mom_21d`)** and **volatility
(`vol_21d`)** as point-in-time signals. The causal-momentum score is a *third,
orthogonal* input:

| Signal | What it captures | Source |
|---|---|---|
| `mom_21d` | a stock's **own** trend | `features.py` |
| `vol_21d` | a stock's **own** risk | `features.py` |
| `causal_mom` | **cross-asset** lead-lag drive from discovered parents | this notebook / POC |

You would register the causal score as just another `@register_signal` so it lands in
the same `features_{version}.parquet`, lagged and lookahead-safe alongside the rest.

---
## 7. Finance-specific pitfalls <a name="pitfalls"></a>

| Pitfall | Why it bites in markets | Mitigation |
|---|---|---|
| **Low signal-to-noise** | Daily returns are ~all noise; true edges are weak. | FDR correction, demand stable edges across refits, size small. |
| **Non-stationarity / regime shifts** | A 2021 graph is wrong in 2022. | Trailing-window refit; consider regime-aware RPCMCI. |
| **Multiple testing** | $N^2\tau_{max}$ tests → guaranteed false edges. | `fdr_bh` (already in the POC); raise $\alpha$ bar for live trading. |
| **Look-ahead via standardization** | Z-scoring with full-sample stats leaks the future. | Standardize within the trailing window only. |
| **Survivorship / point-in-time data** | Delisted names vanish; index membership changes. | Use a survivorship-free universe; freeze membership as of each date. |
| **Latent common causes** | A hidden factor (e.g. liquidity) drives many names → spurious edges. | PCMCI assumes causal sufficiency; add known factors as variables or use LPCMCI. |
| **Microstructure lead-lag** | Asynchronous close times / illiquidity fake 1-day edges. | Align calendars; be skeptical of $\tau=1$ edges between illiquid names. |
| **Overfitting the graph to alpha** | Cherry-picking edges that backtest well. | Decide $\tau_{max}$, $\alpha$, universe *before* looking at PnL. |

---
## 8. Where this plugs into `quantt-pipeline` <a name="pipeline"></a>

Concrete path from this notebook to the production pipeline in `../quantt-pipeline/`:

1. **Ingest** prices via `src/ingest.py` (already long-format, cached parquet).
2. **Pivot to a returns panel** in a new `causal.py` module:
   `np.log(prices).diff()` → trailing window → standardize (the §3 four-liner).
3. **Fit PCMCI per rebalance** on the trailing window; extract `parents_by_target`.
4. **Compute the causal-momentum score** (§5) and register it as a signal:

   ```python
   @register_signal("causal_mom")
   def causal_momentum(panel):
       # fit PCMCI on the trailing window per date, return the per-ticker score
       ...
   ```
5. `build_features()` applies the mandatory `groupby(ticker).shift(1)`, so the causal
   score lands in `features_{version}.parquet` **lookahead-safe** next to `mom_21d`
   and `vol_21d`.

That keeps the whole research stack consistent: one config, one versioned dataset,
one decorator pattern — with causality as just another well-behaved column.

---
### References
- Runge, J. et al. (2019). *Detecting and quantifying causal associations in large
  nonlinear time series datasets.* Science Advances 5(11).
- See [`PCMCI_Intro.ipynb`](./PCMCI_Intro.ipynb) for the algorithm internals.
- `tigramite`: https://github.com/jakobrunge/tigramite